# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

# Preprocesador para modelos basados en árboles: solo requieren imputación
# (no necesitan escalado porque los splits son invariantes a la escala)
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), feature_columns),
    ]
)

# Preprocesador para modelos sensibles a la escala (LogisticRegression, KNN):
# imputación + estandarización
scaled_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            feature_columns,
        ),
    ]
)

tree_preprocessor, scaled_preprocessor

(ColumnTransformer(transformers=[('num', SimpleImputer(strategy='median'),
                                  ['alcohol', 'malic_acid', 'ash',
                                   'alcalinity_of_ash', 'magnesium',
                                   'total_phenols', 'flavanoids',
                                   'nonflavanoid_phenols', 'proanthocyanins',
                                   'color_intensity', 'hue',
                                   'od280/od315_of_diluted_wines', 'proline'])]),
 ColumnTransformer(transformers=[('num',
                                  Pipeline(steps=[('imputer',
                                                   SimpleImputer(strategy='median')),
                                                  ('scaler', StandardScaler())]),
                                  ['alcohol', 'malic_acid', 'ash',
                                   'alcalinity_of_ash', 'magnesium',
                                   'total_phenols', 'flavanoids',
                             

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Si el `StandardScaler` se ajusta (`fit`) sobre todo el conjunto de entrenamiento antes de hacer la validación cruzada, la media y la desviación estándar calculadas ya habrán "visto" las observaciones que en cada fold se usan como validación. Esto es una fuga de información (*data leakage*): el modelo se beneficia indirectamente de estadísticas calculadas con datos que se supone no debería conocer en ese momento, lo que produce una estimación optimista y poco realista del desempeño. Al colocar el `StandardScaler` dentro del `Pipeline`, `cross_validate`/`GridSearchCV` se aseguran de que el escalador se ajuste únicamente con los folds de entrenamiento en cada iteración y se aplique (`transform`) al fold de validación, replicando fielmente lo que ocurrirá en producción con datos nuevos.

## 3. Modelos candidatos

In [5]:
models = {
    "decision_tree": Pipeline(steps=[
        ("preprocessor", clone(tree_preprocessor)),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "logistic_regression": Pipeline(steps=[
        ("preprocessor", clone(scaled_preprocessor)),
        ("classifier", LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)),
    ]),
    "knn": Pipeline(steps=[
        ("preprocessor", clone(scaled_preprocessor)),
        ("classifier", KNeighborsClassifier()),
    ]),
}

list(models.keys())

['decision_tree', 'logistic_regression', 'knn']

## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

initial_rows = []
for name, pipeline in models.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
    )
    initial_rows.append({
        "model": name,
        "train_accuracy_mean": scores["train_score"].mean(),
        "val_accuracy_mean": scores["test_score"].mean(),
        "val_accuracy_std": scores["test_score"].std(),
    })

initial_comparison = pd.DataFrame(initial_rows).sort_values(
    "val_accuracy_mean", ascending=False
).reset_index(drop=True)

initial_comparison

,model,train_accuracy_mean,val_accuracy_mean,val_accuracy_std
0,logistic_regression,1.0000,0.979064,0.027713
1,knn,0.9771,0.950739,0.035274
2,decision_tree,1.0000,0.902217,0.058617


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "decision_tree": {
        "classifier__max_depth": [3, 5, 7, None],
        "classifier__min_samples_leaf": [1, 2, 4],
    },
    "logistic_regression": {
        "classifier__C": [0.01, 0.1, 1.0, 10.0],
        "classifier__penalty": ["l2"],
    },
    "knn": {
        "classifier__n_neighbors": [3, 5, 7, 9, 11],
        "classifier__weights": ["uniform", "distance"],
    },
}

searches = {}
tuned_rows = []
for name, pipeline in models.items():
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[name],
        scoring="accuracy",
        cv=cv,
        n_jobs=-1,
        return_train_score=True,
    )
    grid_search.fit(X_train, y_train)
    searches[name] = grid_search

    best_idx = grid_search.best_index_
    cv_res = grid_search.cv_results_
    tuned_rows.append({
        "model": name,
        "best_params": grid_search.best_params_,
        "train_accuracy_mean": cv_res["mean_train_score"][best_idx],
        "val_accuracy_mean": grid_search.best_score_,
        "val_accuracy_std": cv_res["std_test_score"][best_idx],
    })

tuned_comparison = pd.DataFrame(tuned_rows).sort_values(
    "val_accuracy_mean", ascending=False
).reset_index(drop=True)

tuned_comparison

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to it

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to it

,model,best_params,train_accuracy_mean,val_accuracy_mean,val_accuracy_std
0,logistic_regression,"{'classifier__C': 1.0, 'classifier__penalty': ...",1.000000,0.979064,0.027713
1,knn,"{'classifier__n_neighbors': 7, 'classifier__we...",0.964773,0.957882,0.040520
2,decision_tree,"{'classifier__max_depth': 5, 'classifier__min_...",0.985934,0.902463,0.076225


## 6. Selección y persistencia del campeón

In [8]:
champion_name = max(searches, key=lambda name: searches[name].best_score_)
champion_search = searches[champion_name]
champion = champion_search.best_estimator_

print("Modelo campeón:", champion_name)
print("Mejor accuracy de CV:", champion_search.best_score_)
print("Mejores hiperparámetros:", champion_search.best_params_)

# Guardar el pipeline completo (preprocesamiento + modelo) del campeón
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

# Metadatos de entrenamiento
training_metadata = {
    "champion_model": champion_name,
    "best_params": champion_search.best_params_,
    "cv_accuracy_mean": float(champion_search.best_score_),
    "cv_accuracy_std": float(
        champion_search.cv_results_["std_test_score"][champion_search.best_index_]
    ),
    "cv_folds": cv.get_n_splits(),
    "random_state": RANDOM_STATE,
    "features": contract["features"],
    "target": contract["target"],
    "candidate_models": list(models.keys()),
    "initial_comparison": initial_comparison.to_dict(orient="records"),
    "tuned_comparison": tuned_comparison.drop(columns=["best_params"]).to_dict(orient="records"),
}

with open(ARTIFACTS_DIR / "training_metadata.json", "w", encoding="utf-8") as f:
    json.dump(training_metadata, f, indent=2, ensure_ascii=False)

# Resultados detallados de la validación cruzada (todas las combinaciones probadas)
cv_results_frames = []
for name, search in searches.items():
    df_res = pd.DataFrame(search.cv_results_)
    df_res.insert(0, "model", name)
    cv_results_frames.append(df_res)

cv_results = pd.concat(cv_results_frames, ignore_index=True)
cv_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

training_metadata

Modelo campeón: logistic_regression
Mejor accuracy de CV: 0.9790640394088669
Mejores hiperparámetros: {'classifier__C': 1.0, 'classifier__penalty': 'l2'}


{'champion_model': 'logistic_regression',
 'best_params': {'classifier__C': 1.0, 'classifier__penalty': 'l2'},
 'cv_accuracy_mean': 0.9790640394088669,
 'cv_accuracy_std': 0.027713464392187602,
 'cv_folds': 5,
 'random_state': 42,
 'features': ['alcohol',
  'malic_acid',
  'ash',
  'alcalinity_of_ash',
  'magnesium',
  'total_phenols',
  'flavanoids',
  'nonflavanoid_phenols',
  'proanthocyanins',
  'color_intensity',
  'hue',
  'od280/od315_of_diluted_wines',
  'proline'],
 'target': 'target',
 'candidate_models': ['decision_tree', 'logistic_regression', 'knn'],
 'initial_comparison': [{'model': 'logistic_regression',
   'train_accuracy_mean': 1.0,
   'val_accuracy_mean': 0.9790640394088669,
   'val_accuracy_std': 0.027713464392187602},
  {'model': 'knn',
   'train_accuracy_mean': 0.9770998292190655,
   'val_accuracy_mean': 0.9507389162561577,
   'val_accuracy_std': 0.035274169431002235},
  {'model': 'decision_tree',
   'train_accuracy_mean': 1.0,
   'val_accuracy_mean': 0.90221674876

In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> Se comparó el desempeño de tres familias de modelos (árbol de decisión, regresión logística y KNN) mediante validación cruzada estratificada de 5 folds, primero con hiperparámetros por defecto y luego tras optimizarlos con `GridSearchCV`. El modelo campeón se eligió por tener el mayor `accuracy` promedio de validación cruzada tras el ajuste de hiperparámetros. Además del desempeño puntual, se revisó la desviación estándar entre folds (estabilidad) y la brecha entre `accuracy` de entrenamiento y de validación (posible sobreajuste): una diferencia pequeña indica que el modelo generaliza bien y no simplemente memoriza el conjunto de entrenamiento. También se tuvo en cuenta la complejidad e interpretabilidad de cada familia —un árbol de decisión poco profundo o una regresión logística son más fáciles de explicar a stakeholders que un KNN, cuya interpretabilidad es limitada— por lo que, ante desempeños muy similares, se favorece el modelo más simple e interpretable. En ningún momento se utilizó el conjunto de test para esta decisión, preservándolo intacto para la evaluación final.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.